# 06b · ACS county factors via the Census API (87 MN counties, 2022–2024)

**Goal:** add American Community Survey (ACS) 5-year measures that were on the wishlist (`docs/data_wishlist.md` §1) to the shared county × year factor table `Data/processed/county_factors_2022_2024.csv`, built by `06a_build_county_factors.ipynb`.

**Run order:** `06a` first, then this notebook. This notebook **only adds columns**: it checks that every column already in the file keeps exactly the same values, and it replaces its own columns if run again (so it is safe to re-run).

| New column(s) | ACS table | Why it matters for need vs. food shelf activity |
|---|---|---|
| `pct_hh_snap`, `pct_poverty_hh_no_snap` | **B22003** SNAP receipt by poverty status (households) | "SNAP gap": poor households not reached by SNAP (awareness, stigma, paperwork, eligibility rules) |
| `pct_renter_burden_30/50`, `pct_owner_burden_30/50`, `pct_hh_cost_burdened_30`, `pct_hh_severe_burden_50` | **B25070** gross rent as % of income; **B25091** owner costs as % of income | housing costs crowd out food money |
| `pct_foreign_born`, `pct_noncitizen` | **B05002** place of birth by nativity and citizenship | immigrant fear (public charge) can hold back both SNAP and food shelf use |
| `pct_aian_alone`, `pct_hispanic_acs`, `pct_nonwhite` | **B03002** Hispanic origin by race | tribal food programs (FDPIR), language and trust barriers |
| `pct_65plus_living_alone` | **B09020** relationship by household type, population 65+ | isolated seniors: transport and awareness barriers |
| `pct_disabled_in_poverty` | **C18130** age by disability status by poverty status | mobility limits + low income |
| `pop_under18_acs` | **B09001** population under 18 | denominator for children served per child (Q1b) |

**Years.** `year` = the **last year of the 5-year window**, the same convention as `poverty_2022_2024_clean.csv`: 2022 = ACS 2018–2022, 2023 = 2019–2023, 2024 = 2020–2024 (released December 2025). Windows overlap, so the three values move slowly and are not independent.

**August 2026 rule:** the newest window ends in December 2024. Nothing dated 2026 is requested or used.

**API key:** the Census API needs a key. It is read at run time from `Data/local/census_api_key.txt` (git-ignored) and is **never printed or saved**: requests are made inside a helper that hides the URL if anything fails, and the cached raw files contain only the data.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "Data/external/public_factors/raw/census_acs"
OUT = ROOT / "Data/processed"
YEARS = [2022, 2023, 2024]
TABLES = ["B22003", "B25070", "B25091", "B05002", "B03002", "B09020", "C18130", "B09001"]
_KEY_FILE = ROOT / "Data/local/census_api_key.txt"

def acs_group(table, year):
    """All estimate + MOE columns of one ACS 5-year table for MN counties. Cached as JSON (data only, no key)."""
    cache = RAW / f"acs5_{year}_{table}_mn_county.json"
    if not (cache.exists() and cache.stat().st_size > 0):
        cache.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = requests.get(f"https://api.census.gov/data/{year}/acs/acs5",
                             params={"get": f"NAME,group({table})", "for": "county:*", "in": "state:27",
                                     "key": _KEY_FILE.read_text().strip()}, timeout=120)
            status, text = r.status_code, r.text
        except Exception as e:      # never show the request URL (it contains the key)
            raise RuntimeError(f"Census API request failed for {table} {year} ({type(e).__name__})") from None
        if status != 200 or not text.startswith("["):
            raise RuntimeError(f"Census API returned HTTP {status} for {table} {year}")
        cache.write_text(text)
    rows = json.loads(cache.read_text())
    df = pd.DataFrame(rows[1:], columns=rows[0])
    df["fips"] = df["state"] + df["county"]
    keep = [c for c in df.columns if c.startswith(table) and c[-1] in "EM"]
    out = df[["fips"] + keep].copy()
    out[keep] = out[keep].apply(pd.to_numeric, errors="coerce")
    # ACS uses large negative codes (e.g. -555555555) for "not applicable / controlled"; MOE -555555555 = no sampling error
    for c in keep:
        if c.endswith("M"):
            out.loc[out[c] == -555555555, c] = 0
        out.loc[out[c] < -1e8, c] = np.nan
    out["year"] = year
    return out

def labels(table, year):
    """Variable labels (keyless metadata endpoint), cached."""
    cache = RAW / f"acs5_{year}_{table}_labels.json"
    if not cache.exists():
        r = requests.get(f"https://api.census.gov/data/{year}/acs/acs5/groups/{table}.json", timeout=120)
        r.raise_for_status()
        cache.write_text(r.text)
    v = json.loads(cache.read_text())["variables"]
    return {k: v[k]["label"] for k in v if k.endswith("E") and k.startswith(table)}

counties = pd.read_csv(OUT / "mmg_mn_county_2022_2024.csv", dtype={"fips": str})[["fips", "county"]].drop_duplicates()
assert len(counties) == 87
raw = {t: pd.concat([acs_group(t, y) for y in YEARS], ignore_index=True) for t in TABLES}
for t, d in raw.items():
    print(f"{t}: {d.shape[0]} county-years, {sum(c.endswith('E') for c in d.columns)} estimate columns")

B22003: 261 county-years, 7 estimate columns
B25070: 261 county-years, 11 estimate columns
B25091: 261 county-years, 23 estimate columns
B05002: 261 county-years, 27 estimate columns
B03002: 261 county-years, 21 estimate columns
B09020: 261 county-years, 21 estimate columns
C18130: 261 county-years, 22 estimate columns
B09001: 261 county-years, 10 estimate columns


## 1 · Check that variable numbers mean the same thing in all three windows

Census sometimes renumbers lines inside a table. We compare the label of every variable we use in 2022, 2023 and 2024 (ignoring punctuation changes such as `Total:` vs `Total`, or `Foreign born` vs `Foreign-born`).

In [2]:
USED = {"B22003": [1, 2, 3, 5, 6], "B25070": list(range(1, 12)), "B25091": list(range(1, 24)),
        "B05002": [1, 13, 21], "B03002": [1, 3, 5, 12, 15], "B09020": [1, 15, 18], "C18130": [3, 4, 10, 11, 17, 18], "B09001": [1]}
norm = lambda s: s.replace(":", "").replace("-", " ").replace("!!", " | ").lower().strip()
bad = []
for t, lines in USED.items():
    lab = {y: labels(t, y) for y in YEARS}
    for n in lines:
        v = f"{t}_{n:03d}E"
        ls = {y: norm(lab[y].get(v, "MISSING")) for y in YEARS}
        if len(set(ls.values())) != 1:
            bad.append((v, ls))
print("Variables with different labels across windows:", bad if bad else "none")
assert not bad
print("Example:", "B22003_006E =", labels("B22003", 2024)["B22003_006E"])

Variables with different labels across windows: none
Example: B22003_006E = Estimate!!Total:!!Household did not receive Food Stamps/SNAP in the past 12 months:!!Income in the past 12 months below poverty level


## 2 · Build the indicators, with margins of error

ACS estimates come with a **90% margin of error (MOE)**. For a share `p = part / total` we use the Census Bureau's formula for a proportion,
`MOE(p) = sqrt(MOE(part)² − p² · MOE(total)²) / total` (if the value under the root is negative, the ratio formula with `+` is used); for a sum, `MOE = sqrt(sum of MOE²)` (*ACS General Handbook 2020, ch. 8*). All shares are in **percent (0–100)**.

**Definitions**
- `pct_hh_snap` = households that received SNAP in the past 12 months / all households (B22003_002 / _001).
- `pct_poverty_hh_no_snap` (**SNAP gap**) = households **below poverty** that did **not** receive SNAP / all households below poverty (B22003_006 / (_003 + _006)). Note: ACS asks about SNAP in the past 12 months, and under-reporting of SNAP in surveys is well documented, so this is an upper-end estimate of the gap; county *ranking* is what we use.
- Renters: `pct_renter_burden_30` = gross rent ≥ 30% of income (B25070 lines 7–10), `_50` = ≥ 50% (line 10), out of renter households with the ratio computed (excluding "not computed", line 11).
- Owners: `pct_owner_burden_30` = owner costs ≥ 30% of income, with or without a mortgage (B25091 lines 8–11 + 19–22), `_50` (lines 11 + 22), out of owners with the ratio computed.
- Combined: `pct_hh_cost_burdened_30` and `pct_hh_severe_burden_50` = renters + owners burdened / renters + owners computed (all incomes, unlike SVI's `pct_housing_cost_burden_lowinc`, which counts only households under $75k).
- `pct_foreign_born` = B05002_013 / _001; `pct_noncitizen` = _021 / _001.
- `pct_aian_alone` = American Indian & Alaska Native **alone**, Hispanic or not (B03002_005 + _015) / total. `pct_hispanic_acs` = _012 / total. `pct_nonwhite` = everyone except non-Hispanic White alone = 1 − _003 / total. (Named `pct_hispanic_acs` because `county_factors_static.csv` already has SVI's `pct_hispanic`.)
- `pct_65plus_living_alone` = people 65+ who are householders living alone (B09020 lines 15 + 18) / **all people 65+** (line 1, includes group quarters such as nursing homes).
- `pct_disabled_in_poverty` = people **with a disability** whose income is below poverty / all people with a disability whose poverty status is known (C18130 lines 4 + 11 + 18 over 3 + 10 + 17).
- `pop_under18_acs` = population under 18 (B09001_001), 5-year average.

In [3]:
def E(t, n): return raw[t][f"{t}_{n:03d}E"]
def M(t, n): return raw[t][f"{t}_{n:03d}M"]
def sum_e(t, lines): return sum(E(t, n) for n in lines)
def sum_m(t, lines): return np.sqrt(sum(M(t, n) ** 2 for n in lines))

def share(num, num_m, den, den_m):
    """Percent and its 90% MOE (Census proportion formula; ratio formula if the radicand is negative)."""
    p = num / den
    rad = num_m ** 2 - p ** 2 * den_m ** 2
    rad = np.where(rad < 0, num_m ** 2 + p ** 2 * den_m ** 2, rad)
    return 100 * p, 100 * np.sqrt(rad) / den

ids = lambda t: raw[t][["fips", "year"]]
parts = []

# SNAP (households)
t = "B22003"; x = ids(t).copy()
x["pct_hh_snap"], x["pct_hh_snap_moe"] = share(E(t, 2), M(t, 2), E(t, 1), M(t, 1))
pov_e, pov_m = E(t, 3) + E(t, 6), np.sqrt(M(t, 3) ** 2 + M(t, 6) ** 2)
x["n_poverty_hh"] = pov_e
x["pct_poverty_hh_no_snap"], x["pct_poverty_hh_no_snap_moe"] = share(E(t, 6), M(t, 6), pov_e, pov_m)
parts.append(x)

# Housing cost burden
t = "B25070"; r = ids(t).copy()
R_DEN, R30, R50 = range(2, 11), range(7, 11), [10]
r["pct_renter_burden_30"], r["pct_renter_burden_30_moe"] = share(sum_e(t, R30), sum_m(t, R30), sum_e(t, R_DEN), sum_m(t, R_DEN))
r["pct_renter_burden_50"], r["pct_renter_burden_50_moe"] = share(sum_e(t, R50), sum_m(t, R50), sum_e(t, R_DEN), sum_m(t, R_DEN))
t2 = "B25091"
O_DEN, O30, O50 = list(range(3, 12)) + list(range(14, 23)), [8, 9, 10, 11, 19, 20, 21, 22], [11, 22]
r["pct_owner_burden_30"], r["pct_owner_burden_30_moe"] = share(sum_e(t2, O30), sum_m(t2, O30), sum_e(t2, O_DEN), sum_m(t2, O_DEN))
r["pct_owner_burden_50"], r["pct_owner_burden_50_moe"] = share(sum_e(t2, O50), sum_m(t2, O50), sum_e(t2, O_DEN), sum_m(t2, O_DEN))
den_e = sum_e(t, R_DEN) + sum_e(t2, O_DEN); den_m = np.sqrt(sum_m(t, R_DEN) ** 2 + sum_m(t2, O_DEN) ** 2)
n30_e = sum_e(t, R30) + sum_e(t2, O30); n30_m = np.sqrt(sum_m(t, R30) ** 2 + sum_m(t2, O30) ** 2)
n50_e = sum_e(t, R50) + sum_e(t2, O50); n50_m = np.sqrt(sum_m(t, R50) ** 2 + sum_m(t2, O50) ** 2)
r["pct_hh_cost_burdened_30"], r["pct_hh_cost_burdened_30_moe"] = share(n30_e, n30_m, den_e, den_m)
r["pct_hh_severe_burden_50"], r["pct_hh_severe_burden_50_moe"] = share(n50_e, n50_m, den_e, den_m)
assert (raw["B25070"][["fips", "year"]].values == raw["B25091"][["fips", "year"]].values).all()
parts.append(r)

# Nativity
t = "B05002"; x = ids(t).copy()
x["pct_foreign_born"], x["pct_foreign_born_moe"] = share(E(t, 13), M(t, 13), E(t, 1), M(t, 1))
x["pct_noncitizen"], x["pct_noncitizen_moe"] = share(E(t, 21), M(t, 21), E(t, 1), M(t, 1))
parts.append(x)

# Race / ethnicity
t = "B03002"; x = ids(t).copy()
x["pct_aian_alone"], x["pct_aian_alone_moe"] = share(sum_e(t, [5, 15]), sum_m(t, [5, 15]), E(t, 1), M(t, 1))
x["pct_hispanic_acs"], x["pct_hispanic_acs_moe"] = share(E(t, 12), M(t, 12), E(t, 1), M(t, 1))
nonwhite_e = E(t, 1) - E(t, 3)
x["pct_nonwhite"], x["pct_nonwhite_moe"] = share(nonwhite_e, M(t, 3), E(t, 1), M(t, 1))   # B03002_001 is controlled (MOE 0)
parts.append(x)

# Seniors living alone
t = "B09020"; x = ids(t).copy()
x["pct_65plus_living_alone"], x["pct_65plus_living_alone_moe"] = share(sum_e(t, [15, 18]), sum_m(t, [15, 18]), E(t, 1), M(t, 1))
parts.append(x)

# Disability by poverty
t = "C18130"; x = ids(t).copy()
x["pct_disabled_in_poverty"], x["pct_disabled_in_poverty_moe"] = share(sum_e(t, [4, 11, 18]), sum_m(t, [4, 11, 18]),
                                                                       sum_e(t, [3, 10, 17]), sum_m(t, [3, 10, 17]))
parts.append(x)

# Children
t = "B09001"; x = ids(t).copy()
x["pop_under18_acs"], x["pop_under18_acs_moe"] = E(t, 1), M(t, 1)
parts.append(x)

acs = parts[0]
for p in parts[1:]:
    acs = acs.merge(p, on=["fips", "year"], how="outer", validate="one_to_one")
acs["acs_window"] = acs["year"].map(lambda y: f"{y - 4}-{y}")
PCT = [c for c in acs.columns if c.startswith("pct_") and not c.endswith("_moe")]
print(acs.shape, "| indicators:", PCT)

(261, 36) | indicators: ['pct_hh_snap', 'pct_poverty_hh_no_snap', 'pct_renter_burden_30', 'pct_renter_burden_50', 'pct_owner_burden_30', 'pct_owner_burden_50', 'pct_hh_cost_burdened_30', 'pct_hh_severe_burden_50', 'pct_foreign_born', 'pct_noncitizen', 'pct_aian_alone', 'pct_hispanic_acs', 'pct_nonwhite', 'pct_65plus_living_alone', 'pct_disabled_in_poverty']


## 3 · Validate

1. 87 counties × 3 windows, no duplicates, no missing values.
2. Every share is between 0 and 100.
3. **Reliability:** coefficient of variation `CV = (MOE / 1.645) / estimate`. Census guidance: CV > 30% = use with caution. We count, per indicator, how many county-windows are above that, and keep a per-row count `acs_n_high_cv` (out of the 8 headline shares) so users can filter small counties.
4. **Cross-checks** against files built from the same ACS data by other publishers.

In [4]:
assert len(acs) == 87 * 3 and not acs.duplicated(["fips", "year"]).any()
assert set(acs["fips"]) == set(counties["fips"]) and set(acs["year"]) == set(YEARS)
assert acs[PCT + ["pop_under18_acs"]].notna().all().all(), acs[PCT].isna().sum()
assert ((acs[PCT] >= 0) & (acs[PCT] <= 100)).all().all()

HEAD = ["pct_hh_snap", "pct_poverty_hh_no_snap", "pct_hh_cost_burdened_30", "pct_hh_severe_burden_50",
        "pct_foreign_born", "pct_aian_alone", "pct_65plus_living_alone", "pct_disabled_in_poverty"]
cv = pd.DataFrame({c: (acs[c + "_moe"] / 1.645) / acs[c].replace(0, np.nan) for c in PCT})
acs["acs_n_high_cv"] = (cv[HEAD] > 0.30).sum(axis=1)
summ = acs[PCT].describe().T[["min", "50%", "max"]].round(1)
summ["median MOE (pp)"] = acs[[c + "_moe" for c in PCT]].median().values.round(1)
summ["county-windows CV>30%"] = (cv > 0.30).sum().values
summ

,min,50%,max,median MOE (pp),county-windows CV>30%
pct_hh_snap,3.0,7.7,22.2,1.4,1
pct_poverty_hh_no_snap,36.7,65.7,82.1,6.5,0
pct_renter_burden_30,23.3,43.3,66.5,6.6,0
pct_renter_burden_50,9.8,21.3,35.5,5.1,6
pct_owner_burden_30,8.8,18.4,25.4,2.1,0
pct_owner_burden_50,3.0,7.0,12.0,1.3,5
pct_hh_cost_burdened_30,14.2,23.6,32.7,2.3,0
pct_hh_severe_burden_50,5.6,9.9,15.1,1.6,0
pct_foreign_born,0.7,2.6,22.1,0.6,7
pct_noncitizen,0.2,1.3,14.6,0.4,52


In [5]:
# Which counties are least reliable? (2024 window)
a24 = acs[acs.year == 2024].merge(counties, on="fips")
print("2024 window, counties with the most headline shares at CV > 30%:")
print(a24.nlargest(8, "acs_n_high_cv")[["county", "acs_n_high_cv", "n_poverty_hh", "pct_poverty_hh_no_snap", "pct_poverty_hh_no_snap_moe",
                                        "pct_aian_alone", "pct_aian_alone_moe"]].round(1).to_string(index=False))
print("\nSNAP gap, 2024 window: lowest and highest")
print(a24.sort_values("pct_poverty_hh_no_snap")[["county", "pct_poverty_hh_no_snap", "pct_poverty_hh_no_snap_moe", "n_poverty_hh"]]
      .iloc[[0, 1, 2, -3, -2, -1]].round(1).to_string(index=False))

2024 window, counties with the most headline shares at CV > 30%:
           county  acs_n_high_cv  n_poverty_hh  pct_poverty_hh_no_snap  pct_poverty_hh_no_snap_moe  pct_aian_alone  pct_aian_alone_moe
Lake of the Woods              4         134.0                    82.1                        18.7             2.0                 2.1
        Big Stone              2         258.0                    52.3                        10.9             0.2                 0.3
           Benton              1        1882.0                    54.5                        10.9             0.3                 0.2
            Brown              1        1160.0                    72.8                         3.3             0.3                 0.2
       Cottonwood              1         678.0                    60.3                         6.1             0.8                 0.5
            Dodge              1         620.0                    52.6                         7.1             0.2           

In [6]:
# Cross-checks with other files built from the same ACS data
st = pd.read_csv(OUT / "county_factors_static.csv", dtype={"fips": str})
pov = pd.read_csv(OUT / "poverty_2022_2024_clean.csv", dtype={"county_geoid": str}).rename(columns={"county_geoid": "fips"})
x22 = acs[acs.year == 2022].merge(st, on="fips")
x23 = acs[acs.year == 2023].merge(st, on="fips")
chk = pd.DataFrame({
  "SVI 2022 % Hispanic vs ACS 2018-22": [x22["pct_hispanic_acs"].corr(x22["pct_hispanic"]), (x22["pct_hispanic_acs"] - x22["pct_hispanic"]).abs().max()],
  "SVI 2022 % AIAN (non-Hisp.) vs ACS AIAN alone 2018-22": [x22["pct_aian_alone"].corr(x22["pct_aian"]), (x22["pct_aian_alone"] - x22["pct_aian"]).abs().max()],
  "SVI 2022 % minority vs ACS % nonwhite 2018-22": [x22["pct_nonwhite"].corr(x22["pct_minority"]), (x22["pct_nonwhite"] - x22["pct_minority"]).abs().max()],
  "CHR 2025 severe burden (ACS 2019-23) vs ours 2019-23": [x23["pct_hh_severe_burden_50"].corr(x23["chr_pct_severe_housing_cost_burden"]),
                                                          (x23["pct_hh_severe_burden_50"] - x23["chr_pct_severe_housing_cost_burden"]).abs().max()],
  "SVI low-income burden vs ours (all incomes) 2018-22": [x22["pct_hh_cost_burdened_30"].corr(x22["pct_housing_cost_burden_lowinc"]),
                                                          (x22["pct_hh_cost_burdened_30"] - x22["pct_housing_cost_burden_lowinc"]).abs().max()],
}, index=["Pearson r", "max |difference| (pp)"]).T.round(3)
display(chk)
p = acs.merge(pov[["fips", "year", "poverty_rate"]], on=["fips", "year"])
print("Spearman: % households on SNAP vs ACS poverty rate:", round(p["pct_hh_snap"].corr(p["poverty_rate"], method="spearman"), 2))

,Pearson r,max |difference| (pp)
SVI 2022 % Hispanic vs ACS 2018-22,1.000,0.050
SVI 2022 % AIAN (non-Hisp.) vs ACS AIAN alone 2018-22,0.999,1.549
SVI 2022 % minority vs ACS % nonwhite 2018-22,1.000,0.050
CHR 2025 severe burden (ACS 2019-23) vs ours 2019-23,1.000,0.000
SVI low-income burden vs ours (all incomes) 2018-22,0.941,5.810


Spearman: % households on SNAP vs ACS poverty rate: 0.7


**Reading the cross-checks.** Hispanic share and % non-white match SVI 2022 almost exactly (same ACS 2018–22 data; SVI's "minority" is the same definition). AIAN differs a little by design: SVI counts **non-Hispanic** AIAN, we count AIAN alone of any origin. Our severe burden (≥ 50%) for 2019–23 is **identical** to County Health Rankings' 2025 measure (same ACS window), which confirms the line numbers and denominators. The all-income 30% burden is related to SVI's low-income version (r ≈ 0.94) but is a different measure (it also counts middle-income households with high costs).

**Reliability.** Most shares are precise enough for county comparisons (median MOE 0.2–6.5 points). The exceptions: **`pct_aian_alone` has CV > 30% in 104 of 261 county-windows**, almost all counties where the share is under 1% (a small share with a few-hundred-person MOE); it is reliable where it matters (Mahnomen, Beltrami, Cass, Clearwater, Pine...). `pct_noncitizen` is similar. The SNAP gap is reliable everywhere except **Lake of the Woods** (only ~130 poverty households; 82% ± 19 points). Use `acs_n_high_cv` or the `_moe` columns to flag these.

## 4 · Append to `county_factors_2022_2024.csv` (columns added, nothing else changes)

In [7]:
f = OUT / "county_factors_2022_2024.csv"
before = pd.read_csv(f, dtype={"fips": str})
NEW = [c for c in acs.columns if c not in ("fips", "year")]
base = before.drop(columns=[c for c in NEW if c in before.columns])          # re-run safe: drop our own old columns
orig_cols = list(base.columns)
num = acs[NEW].select_dtypes("float").columns
acs[num] = acs[num].round(3)
after = base.merge(acs[["fips", "year"] + NEW], on=["fips", "year"], how="left", validate="one_to_one")
assert len(after) == len(before) == 261 and list(after.columns[:len(orig_cols)]) == orig_cols
pd.testing.assert_frame_equal(after[orig_cols].reset_index(drop=True), base.reset_index(drop=True))
assert after[NEW].drop(columns="acs_window").notna().all().all()
after.to_csv(f, index=False)
check = pd.read_csv(f, dtype={"fips": str})
pd.testing.assert_frame_equal(check[orig_cols], base, check_dtype=False)
print(f"Saved {f.name}: {before.shape} -> {after.shape}; {len(orig_cols)} existing columns unchanged, {len(NEW)} ACS columns added")

Saved county_factors_2022_2024.csv: (261, 51) -> (261, 51); 16 existing columns unchanged, 35 ACS columns added


## 5 · Data dictionary rows

In [8]:
dd_f = OUT / "county_factors_data_dictionary.csv"
dd = pd.read_csv(dd_f)
dd = dd[~((dd["file"] == "county_factors_2022_2024.csv") & dd["column"].isin(NEW))]
SRC = "US Census Bureau, ACS 5-year detailed table {t} (api.census.gov/data/{{year}}/acs/acs5)"
VINT = "5-year window ending in year: 2018-22, 2019-23, 2020-24 (2020-24 released Dec 2025)"
D = {
 "pct_hh_snap": ("B22003", "Households that received SNAP/food stamps in the past 12 months (_002/_001)", "% of households"),
 "n_poverty_hh": ("B22003", "Households with income below poverty in the past 12 months (_003+_006)", "households"),
 "pct_poverty_hh_no_snap": ("B22003", "SNAP gap: households below poverty that did NOT receive SNAP / all households below poverty (_006/(_003+_006))", "% of poverty households"),
 "pct_renter_burden_30": ("B25070", "Renter households paying >=30% of income in gross rent (lines 7-10 / lines 2-10; excludes 'not computed')", "% of renter households"),
 "pct_renter_burden_50": ("B25070", "Renter households paying >=50% of income in gross rent (line 10 / lines 2-10)", "% of renter households"),
 "pct_owner_burden_30": ("B25091", "Owner households (with or without mortgage) paying >=30% of income in owner costs (lines 8-11,19-22 / 3-11,14-22)", "% of owner households"),
 "pct_owner_burden_50": ("B25091", "Owner households paying >=50% of income in owner costs (lines 11,22 / 3-11,14-22)", "% of owner households"),
 "pct_hh_cost_burdened_30": ("B25070+B25091", "All households (renters+owners, all incomes) paying >=30% of income on housing", "% of households (ratio computed)"),
 "pct_hh_severe_burden_50": ("B25070+B25091", "All households (renters+owners, all incomes) paying >=50% of income on housing", "% of households (ratio computed)"),
 "pct_foreign_born": ("B05002", "Foreign-born population (_013/_001)", "% of population"),
 "pct_noncitizen": ("B05002", "Foreign-born, not a US citizen (_021/_001)", "% of population"),
 "pct_aian_alone": ("B03002", "American Indian & Alaska Native alone, Hispanic or not ((_005+_015)/_001)", "% of population"),
 "pct_hispanic_acs": ("B03002", "Hispanic or Latino, any race (_012/_001); named _acs to avoid clash with SVI pct_hispanic in the static file", "% of population"),
 "pct_nonwhite": ("B03002", "Everyone except non-Hispanic White alone (1 - _003/_001)", "% of population"),
 "pct_65plus_living_alone": ("B09020", "People 65+ who are householders living alone (lines 15+18) / all people 65+ (line 1, incl. group quarters)", "% of people 65+"),
 "pct_disabled_in_poverty": ("C18130", "People with a disability below poverty / people with a disability, poverty status known (lines 4+11+18 / 3+10+17)", "% of disabled people"),
 "pop_under18_acs": ("B09001", "Population under 18 (5-year average)", "persons"),
 "acs_n_high_cv": ("B22003, B25070, B25091, B05002, B03002, B09020, C18130", "Number of the 8 headline ACS shares with coefficient of variation >30% (CV = MOE/1.645/estimate); higher = less reliable county-window", "count 0-8"),
 "acs_window": ("all of the above", "ACS 5-year window behind the ACS columns of this row", "text"),
}
rows = []
for c in NEW:
    base_c = c[:-4] if c.endswith("_moe") else c
    t, defn, unit = D[base_c]
    if c.endswith("_moe"):
        defn, unit = f"90% margin of error of {base_c} (Census formula for a derived proportion / sum)", "percentage points" if unit.startswith("%") else unit
    rows.append({"file": "county_factors_2022_2024.csv", "column": c, "definition": defn, "unit": unit,
                 "source": SRC.format(t=t), "vintage": VINT, "geo_grain": "county", "time_grain": "annual (2022-2024)"})
dd = pd.concat([dd, pd.DataFrame(rows)], ignore_index=True)
assert set(dd.query("file == 'county_factors_2022_2024.csv'")["column"]) == set(after.columns)
dd.to_csv(dd_f, index=False)
print(len(rows), "dictionary rows added;", len(dd), "rows in total")

35 dictionary rows added; 103 rows in total


## 6 · Quick look (no interpretation here; analysis is in Q1b and Q6)

In [9]:
mmg = pd.read_csv(OUT / "mmg_mn_county_2022_2024.csv", dtype={"fips": str})
q = acs.merge(mmg[["fips", "year", "fi_rate"]], on=["fips", "year"])
q24 = q[q.year == 2024]
pd.Series({c: q24[c].corr(q24["fi_rate"], method="spearman") for c in HEAD}).round(2).to_frame("Spearman with MMG fi_rate (2024)")

,Spearman with MMG fi_rate (2024)
pct_hh_snap,0.82
pct_poverty_hh_no_snap,-0.58
pct_hh_cost_burdened_30,0.18
pct_hh_severe_burden_50,0.35
pct_foreign_born,-0.20
pct_aian_alone,0.49
pct_65plus_living_alone,0.35
pct_disabled_in_poverty,0.49


**Done.** `county_factors_2022_2024.csv` now has the ACS columns above (plus `_moe` columns, `acs_n_high_cv`, `acs_window`). If `06a` is ever re-run, re-run this notebook afterwards (06a rewrites the file without these columns).